In [10]:
!pip install -q scipy pandas plotly matplotlib

In [11]:
!pip install -q mediapipe torch torchvision scipy pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 3.8 MB/s eta 0:00:00


In [5]:
# 셀 2 아래에 추가
!wget -q -O "{MODEL_DIR}/model.py" \
  "https://raw.githubusercontent.com/wmcnally/golfdb/master/model.py"
print("✅ model.py 다운로드 완료")

✅ model.py 다운로드 완료


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
BASE_DIR  = '/content/drive/MyDrive/golf_project'
MODEL_DIR = f'{BASE_DIR}/models'
DATA_DIR = f'{BASE_DIR}/data/processed'
sys.path.insert(0, MODEL_DIR)

for fname in ['pose_landmarker_heavy.task', 'swingnet_1800.pth.tar']:
    path = f'{MODEL_DIR}/{fname}'
    ok   = os.path.exists(path)
    sz   = f'{os.path.getsize(path)/1e6:.1f}MB' if ok else '없음'
    print(f'{"✅" if ok else "❌"} {fname} ({sz})')

Mounted at /content/drive
✅ pose_landmarker_heavy.task (30.7MB)
✅ swingnet_1800.pth.tar (63.3MB)


In [12]:
import cv2, json, math, numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

import mediapipe as mp

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

import torch, torchvision
from torchvision.transforms import functional as TF

try:
    from model import EventDetector
    SWINGNET_OK = True
    print("✅ SwingNet import 성공")
except ImportError:
    SWINGNET_OK = False
    print("⚠️ SwingNet 없음 → Fallback 사용")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

MPP_VIS_THR    = 0.5
RCNN_SCORE_THR = 0.4
SEQ_LEN        = 64
BATCH_SIZE     = 16

MPP_33 = [
    'nose','left_eye_inner','left_eye','left_eye_outer',
    'right_eye_inner','right_eye','right_eye_outer',
    'left_ear','right_ear','mouth_left','mouth_right',
    'left_shoulder','right_shoulder',
    'left_elbow','right_elbow',
    'left_wrist','right_wrist',
    'left_pinky','right_pinky',
    'left_index','right_index',
    'left_thumb','right_thumb',
    'left_hip','right_hip',
    'left_knee','right_knee',
    'left_ankle','right_ankle',
    'left_heel','right_heel',
    'left_foot_index','right_foot_index',
]
ALL_34 = MPP_33 + ['hip_center']

FUSION_TARGETS = {
    'left_wrist','right_wrist','left_elbow','right_elbow',
    'left_knee','right_knee','left_ankle','right_ankle',
    'left_heel','right_heel','left_foot_index','right_foot_index',
}
COCO17_TO_MPP = {
    0:'nose', 1:'left_eye', 2:'right_eye', 3:'left_ear', 4:'right_ear',
    5:'left_shoulder', 6:'right_shoulder', 7:'left_elbow', 8:'right_elbow',
    9:'left_wrist', 10:'right_wrist', 11:'left_hip', 12:'right_hip',
    13:'left_knee', 14:'right_knee', 15:'left_ankle', 16:'right_ankle',
}
EVENT_NAMES = [
    'address','toe_up','mid_backswing','top',
    'mid_downswing','impact','mid_follow_through','finish'
]
print("✅ 완료")

⚠️ SwingNet 없음 → Fallback 사용
Device: cuda
✅ 완료


In [8]:
def load_mediapipe(path):
    opts = mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=path),
        output_segmentation_masks=False,
        num_poses=1,
        min_pose_detection_confidence=0.3,
        min_pose_presence_confidence=0.3,
        min_tracking_confidence=0.3,
    )
    return mp_vision.PoseLandmarker.create_from_options(opts)

def load_rcnn(device):
    m = torchvision.models.detection.keypointrcnn_resnet50_fpn(
        weights=torchvision.models.detection.KeypointRCNN_ResNet50_FPN_Weights.DEFAULT
    )
    return m.to(device).eval()

def load_swingnet(path, device):
    if not SWINGNET_OK:
        return None
    m = EventDetector(pretrain=False, width_mult=1.0,
                      lstm_layers=1, lstm_hidden=256,
                      bidirectional=True, dropout=False)
    state = torch.load(path, map_location=device)
    m.load_state_dict(state.get('model_state_dict', state))
    return m.to(device).eval()

print("MediaPipe 로드 중...")
landmarker = load_mediapipe(f'{MODEL_DIR}/pose_landmarker_heavy.task')
print("✅ MediaPipe")

try:
    rcnn_model = load_rcnn(DEVICE)
    print("✅ R-CNN")
except Exception as e:
    rcnn_model = None
    print(f"⚠️ R-CNN 실패: {e}")

try:
    swingnet = load_swingnet(f'{MODEL_DIR}/swingnet_1800.pth.tar', DEVICE)
    print("✅ SwingNet")
except Exception as e:
    swingnet = None
    print(f"⚠️ SwingNet 실패: {e}")

MediaPipe 로드 중...
✅ MediaPipe
Downloading: "https://download.pytorch.org/models/keypointrcnn_resnet50_fpn_coco-fc266e95.pth" to /root/.cache/torch/hub/checkpoints/keypointrcnn_resnet50_fpn_coco-fc266e95.pth


100%|██████████| 226M/226M [00:01<00:00, 177MB/s]


✅ R-CNN
✅ SwingNet


In [9]:
def mpp_infer(landmarker, frame_rgb):
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    det = landmarker.detect(mp_img)
    if not det.pose_landmarks:
        return False, []
    pose_lms = det.pose_landmarks[0]
    world_lms = det.pose_world_landmarks[0] if det.pose_world_landmarks else None
    lms = []
    for i, lm in enumerate(pose_lms):
        lms.append({
            'name': MPP_33[i],
            'x': float(lm.x),
            'y': float(lm.y),
            'z': float(world_lms[i].z if world_lms else lm.z),
            'visibility': float(getattr(lm, 'visibility', 1.0)),
            'source': 'mediapipe',
            'aux_score': 0.0,
        })
    lh, rh = pose_lms[23], pose_lms[24]
    lhw, rhw = (world_lms[23], world_lms[24]) if world_lms else (lh, rh)
    lms.append({
        'name': 'hip_center',
        'x': (float(lh.x) + float(rh.x)) / 2,
        'y': (float(lh.y) + float(rh.y)) / 2,
        'z': (float(lhw.z) + float(rhw.z)) / 2,
        'visibility': min(float(getattr(lh,'visibility',1.0)),
                         float(getattr(rh,'visibility',1.0))),
        'source': 'mediapipe',
        'aux_score': 0.0,
    })
    return True, lms


def rcnn_infer(model, frame_bgr, H, W):
    img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    tensor = TF.to_tensor(img_rgb).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        preds = model(tensor)[0]
    if len(preds['scores']) == 0:
        return {}
    best = preds['scores'].argmax().item()
    if preds['scores'][best] < RCNN_SCORE_THR:
        return {}
    kps = preds['keypoints'][best].cpu().numpy()
    return {COCO17_TO_MPP[i]: {'x': float(kps[i,0])/W,
                                'y': float(kps[i,1])/H,
                                'score': float(kps[i,2])}
            for i in COCO17_TO_MPP}


def fuse(mpp_lms, rcnn_res):
    fused, cnt = [], 0
    for lm in mpp_lms:
        lm_c = dict(lm)
        name = lm['name']
        if (name in FUSION_TARGETS
                and name in rcnn_res
                and lm['visibility'] < MPP_VIS_THR
                and rcnn_res[name]['score'] >= RCNN_SCORE_THR):
            lm_c.update({'x': rcnn_res[name]['x'], 'y': rcnn_res[name]['y'],
                         'source': 'rcnn', 'aux_score': rcnn_res[name]['score']})
            cnt += 1
        fused.append(lm_c)
    return fused, cnt


def detect_events(swingnet, frames_np, fps):
    """
    ✅ 수정:
    - frames_np: uint8 → float32 변환은 호출 직전에만 (메모리 절약)
    - batch_size 16 → 4 (GPU 메모리 부담 감소)
    - probs /= SEQ_LEN (버그) → 실제 누적 카운트 기반 평균으로 수정
    """
    n = len(frames_np)
    if swingnet is None or n < SEQ_LEN:
        return None

    probs  = torch.zeros(n, len(EVENT_NAMES))
    counts = torch.zeros(n, 1)

    for start in range(0, n - SEQ_LEN + 1, 4):   # ✅ batch_size 16 → 4
        end   = min(start + 4, n - SEQ_LEN + 1)
        batch = [torch.FloatTensor(frames_np[i:i+SEQ_LEN]).permute(0,3,1,2)
                 for i in range(start, end)]
        bt = torch.stack(batch).to(DEVICE)
        with torch.no_grad():
            out = swingnet(bt.view(-1, 3, 160, 160))
            out = out.view(len(batch), SEQ_LEN, -1)
        for b, i in enumerate(range(start, end)):
            probs[i:i+SEQ_LEN]  += out[b].cpu()
            counts[i:i+SEQ_LEN] += 1               # ✅ 실제 카운트 누적

    probs /= counts.clamp(min=1)                    # ✅ 고정값 나눔 버그 수정

    events = {}
    for e_idx, name in enumerate(EVENT_NAMES):
        fi   = int(probs[:, e_idx].argmax().item())
        conf = float(probs[fi, e_idx].item())
        events[name] = {'frame': fi, 'confidence': round(conf, 4),
                        'timestamp': round(fi / fps, 4)}
    return events


def fallback_events(frames, fps):
    wy = []
    for f in frames:
        lm = next((l for l in f.get('landmarks',[]) if l['name']=='left_wrist'), None)
        wy.append(lm['y'] if lm else np.nan)
    wy = np.array(wy, dtype=float); n = len(wy)
    def mk(fi): return {'frame':int(fi),'confidence':0.0,'method':'fallback',
                        'timestamp':round(fi/fps,4)}
    addr   = int(n * 0.08)
    top_f  = int(np.nanargmin(wy[addr:]) + addr)
    grad   = np.gradient(np.nan_to_num(wy))
    impact = int(np.argmax(grad[top_f:]) + top_f)
    finish = min(impact + int(fps*0.5), n-1)
    mid_bs = (addr + top_f) // 2
    mid_ds = (top_f + impact) // 2
    mid_ft = (impact + finish) // 2
    toe_up = (addr + mid_bs) // 2
    return {
        'address': mk(addr),       'toe_up': mk(toe_up),
        'mid_backswing': mk(mid_bs), 'top': mk(top_f),
        'mid_downswing': mk(mid_ds), 'impact': mk(impact),
        'mid_follow_through': mk(mid_ft), 'finish': mk(finish),
    }

print("✅ 완료")

✅ 완료


In [10]:
def process_video_step24(video_path, output_base, view_type):
    out_dir = Path(output_base) / 'step24_events' / view_type
    out_dir.mkdir(parents=True, exist_ok=True)
    video_name = Path(video_path).stem
    out_path   = out_dir / f"{video_name}_events.json"

    # ✅ 기존: out_path.exists() → 무조건 스킵 (버그)
    # ✅ 수정: 파일이 있어도 'events' 키가 없으면 불완전 → 재처리
    if out_path.exists():
        try:
            with open(out_path, 'r') as f:
                existing = json.load(f)
            if existing.get('events') and existing.get('frames'):
                print(f"  ⏭  스킵 (정상 완료): {video_name}")
                return str(out_path)
            else:
                print(f"  ♻️  재처리 (불완전 파일): {video_name}")
                out_path.unlink()   # 불완전 파일 삭제 후 재처리
        except Exception:
            print(f"  ♻️  재처리 (손상 파일): {video_name}")
            out_path.unlink()

    cap   = cv2.VideoCapture(str(video_path))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    frames          = []
    swingnet_frames = []   # ✅ uint8 유지 (float32 변환 최대한 지연)
    pose_count = fused_total = 0

    for fi in tqdm(range(total), desc=video_name, leave=False):
        ret, frame_bgr = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        # ✅ uint8 그대로 append (기존: 매 프레임마다 float32 변환)
        swingnet_frames.append(cv2.resize(frame_rgb, (160, 160)))

        has_pose, mpp_lms = mpp_infer(landmarker, frame_rgb)

        fused_cnt = 0
        if has_pose and rcnn_model is not None:
            rcnn_res = rcnn_infer(rcnn_model, frame_bgr, H, W)
            final_lms, fused_cnt = fuse(mpp_lms, rcnn_res)
        else:
            final_lms = mpp_lms

        pose_count  += int(has_pose)
        fused_total += fused_cnt
        frames.append({
            'frame':     fi,
            'timestamp': round(fi / fps, 6),
            'has_pose':  has_pose,
            'landmarks': final_lms,
        })

    cap.release()

    # ✅ float32 변환: 필요 시점에만, 직후 즉시 해제
    frames_np = np.array(swingnet_frames, dtype=np.float32) / 255.0
    del swingnet_frames

    events = detect_events(swingnet, frames_np, fps)
    del frames_np

    if events is None:
        print(f"  ⚠️ SwingNet 실패 → Fallback 사용")
        events = fallback_events(frames, fps)
    else:
        print(f"  📍 이벤트: { {k:v['frame'] for k,v in events.items()} }")

    # ✅ indent 제거 (대용량 파일 저장 속도 향상)
    with open(out_path, 'w') as f:
        json.dump({
            'video':      video_name,
            'view_type':  view_type,
            'original_size': {'width': W, 'height': H},
            'fps':        fps,
            'total_frames': len(frames),
            'frames_with_pose': pose_count,
            'landmark_count': 34,
            'landmark_names': ALL_34,
            'extraction': {
                'mpp_model':           'pose_landmarker_heavy',
                'aux_model':           'keypoint_rcnn' if rcnn_model else 'none',
                'event_model':         'swingnet_1800' if swingnet else 'fallback',
                'mpp_vis_threshold':   MPP_VIS_THR,
                'aux_score_threshold': RCNN_SCORE_THR,
                'total_fused':         fused_total,
            },
            'events': events,
            'frames': frames,
        }, f)

    print(f"  ✅ {video_name} | 포즈:{pose_count}/{len(frames)} | RCNN보완:{fused_total}")
    return str(out_path)

print("✅ 완료")

✅ 완료


In [11]:
import math
import json
import shutil
from pathlib import Path
from google.colab import drive

# ── Drive 마운트 ──────────────────────────────────────────────
drive.mount('/content/drive', force_remount=False)
DRIVE_CKPT_DIR = Path('/content/drive/MyDrive/golf_pipeline/checkpoints')
DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ✅ 로컬 경로 (빠른 I/O)
LOCAL_CKPT_DIR = Path('/content/golf_pipeline_ckpt')
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_CKPT    = LOCAL_CKPT_DIR / 'step24_done.json'     # 로컬 완료 목록
LOCAL_RESULT  = LOCAL_CKPT_DIR / 'step24_results.jsonl' # ✅ JSONL — 한 줄씩 append

DRIVE_CKPT    = DRIVE_CKPT_DIR / 'step24_done.json'     # Drive 백업 (체크포인트만)
DRIVE_RESULT  = DRIVE_CKPT_DIR / 'step24_results.jsonl' # Drive 백업

DRIVE_SYNC_INTERVAL = 25  # ✅ 25개마다 Drive 백업 (5 → 25)

# ── 체크포인트 로드 (Drive → 로컬 우선순위) ─────────────────
#    로컬이 없으면 Drive에서 가져옴
if not LOCAL_CKPT.exists() and DRIVE_CKPT.exists():
    shutil.copy(DRIVE_CKPT,   LOCAL_CKPT)
    shutil.copy(DRIVE_RESULT, LOCAL_RESULT)
    print("♻️  Drive → 로컬 복원 완료")

if LOCAL_CKPT.exists():
    with open(LOCAL_CKPT, 'r') as f:
        done_set = set(json.load(f))
    print(f"♻️  체크포인트 로드: {len(done_set)}개 완료 기록")
else:
    done_set = set()
    print("🆕 체크포인트 없음 — 처음부터 시작")

# ── 대상 영상 수집 (뷰별 20%) ─────────────────────────────────
step1_dir = Path(DATA_DIR) / 'step1_square_padded'
assert step1_dir.exists(), f"❌ step1_padded 없음: {step1_dir}"

video_files = []
for view_dir in sorted(step1_dir.iterdir()):
    if not view_dir.is_dir():
        continue
    all_mp4 = sorted(view_dir.glob('*.mp4'))
    n       = max(1, math.ceil(len(all_mp4) * 0.2))
    picked  = all_mp4[:n]
    pending = [mp4 for mp4 in picked if mp4.stem not in done_set]
    print(f"  {view_dir.name}: 전체 {len(all_mp4)}개 → 대상 {n}개 "
          f"(완료 {n - len(pending)}개 / 남은 {len(pending)}개)")
    for mp4 in picked:
        video_files.append({'path': mp4, 'view_type': view_dir.name})

pending_files = [vf for vf in video_files if vf['path'].stem not in done_set]
total = len(video_files)
print(f"\n총 {total}개 중 {len(pending_files)}개 처리 시작\n")

# ── 처리 루프 ─────────────────────────────────────────────────
def sync_to_drive():
    """로컬 → Drive 백업 (체크포인트 + 결과)"""
    with open(LOCAL_CKPT, 'w') as f:
        json.dump(list(done_set), f)
    shutil.copy(LOCAL_CKPT,   DRIVE_CKPT)
    if LOCAL_RESULT.exists():
        shutil.copy(LOCAL_RESULT, DRIVE_RESULT)
    print(f"   💾 Drive 백업 ({len(done_set)}개 완료)")

ok_count  = 0
err_count = 0

for i, vf in enumerate(pending_files):
    stem = vf['path'].stem
    idx  = total - len(pending_files) + i + 1
    print(f"[{idx}/{total}] {stem}  [{vf['view_type']}]")

    try:
        out = process_video_step24(vf['path'], DATA_DIR, vf['view_type'])

        # ✅ 결과는 로컬 JSONL에 한 줄씩 append (누적 덮어쓰기 X)
        record = {'video': stem, 'view': vf['view_type'], 'status': 'ok',
                  'output_path': str(out)}   # out 전체 대신 경로만 저장
        with open(LOCAL_RESULT, 'a', encoding='utf-8') as f:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

        done_set.add(stem)
        ok_count += 1

    except Exception as e:
        import traceback; traceback.print_exc()
        record = {'video': stem, 'view': vf['view_type'], 'status': 'error', 'error': str(e)}
        with open(LOCAL_RESULT, 'a', encoding='utf-8') as f:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

        done_set.add(stem)  # 에러도 완료 처리 (무한 재시도 방지)
        err_count += 1

    # ✅ 로컬 체크포인트는 매번 (빠름), Drive는 25개마다만
    with open(LOCAL_CKPT, 'w') as f:
        json.dump(list(done_set), f)

    if (i + 1) % DRIVE_SYNC_INTERVAL == 0:
        sync_to_drive()

# ── 최종 Drive 동기화 ─────────────────────────────────────────
sync_to_drive()

print(f"\n{'='*50}")
print(f"✅ Step 2+4 완료: {ok_count} 성공  |  실패 {err_count}개")
print(f"💾 로컬: {LOCAL_RESULT}")
print(f"💾 Drive: {DRIVE_RESULT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🆕 체크포인트 없음 — 처음부터 시작
  dtl: 전체 585개 → 대상 117개 (완료 0개 / 남은 117개)
  face_on: 전체 461개 → 대상 93개 (완료 0개 / 남은 93개)
  other: 전체 354개 → 대상 71개 (완료 0개 / 남은 71개)

총 281개 중 281개 처리 시작

[1/281] 0_square  [dtl]


0_square:   0%|          | 0/138 [00:00<?, ?it/s]

  ⚠️ SwingNet 실패 → Fallback 사용
  ✅ 0_square | 포즈:134/138 | RCNN보완:339
[2/281] 1000_square  [dtl]


1000_square:   0%|          | 0/392 [00:00<?, ?it/s]

  ⚠️ SwingNet 실패 → Fallback 사용
  ✅ 1000_square | 포즈:392/392 | RCNN보완:1166
[3/281] 1001_square  [dtl]


...

  ⚠️ SwingNet 실패 → Fallback 사용
  ✅ 1219_square | 포즈:267/267 | RCNN보완:423
[280/281] 1220_square  [other]


1220_square:   0%|          | 0/282 [00:00<?, ?it/s]

  ⚠️ SwingNet 실패 → Fallback 사용
  ✅ 1220_square | 포즈:282/282 | RCNN보완:411
[281/281] 1233_square  [other]


1233_square:   0%|          | 0/464 [00:00<?, ?it/s]

  ⚠️ SwingNet 실패 → Fallback 사용
  ✅ 1233_square | 포즈:464/464 | RCNN보완:1171
   💾 Drive 백업 (281개 완료)

✅ Step 2+4 완료: 281 성공  |  실패 0개
💾 로컬: /content/golf_pipeline_ckpt/step24_results.jsonl
💾 Drive: /content/drive/MyDrive/golf_pipeline/checkpoints/step24_results.jsonl


In [5]:
import pandas as pd
from scipy.signal import savgol_filter
from scipy.interpolate import CubicSpline

Z_SCALE   = 0.3
VIS_THR3  = 0.5
SIG_XY    = 2.5
SIG_Z     = 3.0
WIN       = 15
BONE_TOL  = 0.15
MAX_GAP   = 8     # 연속 NaN 최대 보간 프레임 (freeze 방지)

OCCLUSION_PRONE = {
    'left_wrist','right_wrist','left_elbow','right_elbow',
    'left_pinky','right_pinky','left_index','right_index',
    'left_thumb','right_thumb','left_knee','right_knee',
    'left_ankle','right_ankle','left_heel','right_heel',
    'left_foot_index','right_foot_index',
}
BONE_PAIRS = [
    ('left_shoulder','left_elbow'),    ('left_elbow','left_wrist'),
    ('left_wrist','left_index'),       ('right_shoulder','right_elbow'),
    ('right_elbow','right_wrist'),     ('right_wrist','right_index'),
    ('left_hip','left_knee'),          ('left_knee','left_ankle'),
    ('left_ankle','left_heel'),        ('left_ankle','left_foot_index'),
    ('right_hip','right_knee'),        ('right_knee','right_ankle'),
    ('right_ankle','right_heel'),      ('right_ankle','right_foot_index'),
    ('left_shoulder','right_shoulder'),('left_hip','right_hip'),
]
print("✅ 완료")

✅ 완료


In [6]:
def cubic_spline_interp(frames):
    records = []
    for f in frames:
        row = {}
        if f.get('has_pose') and f.get('landmarks'):
            for lm in f['landmarks']:
                name, vis = lm['name'], lm.get('visibility', 1.0)
                if name in OCCLUSION_PRONE and vis < VIS_THR3:
                    row[f'{name}_x'] = row[f'{name}_y'] = row[f'{name}_z'] = np.nan
                else:
                    row[f'{name}_x'] = lm['x']
                    row[f'{name}_y'] = lm['y']
                    row[f'{name}_z'] = lm['z']
        else:
            for name in ALL_34:
                row[f'{name}_x'] = row[f'{name}_y'] = row[f'{name}_z'] = np.nan
        records.append(row)

    df   = pd.DataFrame(records)
    cols = [c for c in df.columns if c.endswith(('_x','_y','_z'))]
    df[cols] = df[cols].interpolate(method='cubic', limit_direction='both', limit=MAX_GAP)
    df[cols] = df[cols].ffill(limit=3).bfill(limit=3)

    for i, f in enumerate(frames):
        if not f.get('has_pose') or not f.get('landmarks'):
            continue
        for lm in f['landmarks']:
            for ax in ('x','y','z'):
                v = df.at[i, f"{lm['name']}_{ax}"]
                if pd.notna(v):
                    lm[ax] = float(v)
    return frames


def fix_outliers(frames, lm_name, axis, sigma):
    values = []
    for f in frames:
        lm = next((l for l in f.get('landmarks',[]) if l['name']==lm_name), None)
        values.append(lm.get(axis, np.nan) if lm else np.nan)
    values = np.array(values, dtype=float)
    valid  = ~np.isnan(values)
    if valid.sum() < 5:
        return frames, 0
    w = max(min(WIN, valid.sum()), 5)
    w = w if w%2==1 else w-1
    smoothed        = values.copy()
    smoothed[valid] = savgol_filter(values[valid], w, 3)
    res  = np.abs(values - smoothed)
    vres = res[valid]
    mad  = np.median(np.abs(vres - np.median(vres)))
    rsig = mad * 1.4826
    if rsig < 1e-8:
        return frames, 0
    outlier = (res > sigma * rsig) & valid
    idxs = np.where(outlier)[0]
    i = 0
    while i < len(idxs):
        s = idxs[i]; e = s; j = i+1
        while j < len(idxs) and idxs[j]==idxs[j-1]+1:
            e = idxs[j]; j += 1
        if e - s + 1 > 5:
            outlier[s:e+1] = False
        i = j
    if valid.sum()>0 and outlier.sum()/valid.sum() > 0.3:
        return frames, 0
    fixed = values.copy()
    fixed[outlier] = np.nan
    nanm = np.isnan(fixed)
    if nanm.any() and (~nanm).sum() >= 2:
        xv, yv = np.where(~nanm)[0], fixed[~nanm]
        cs = CubicSpline(xv, yv, extrapolate=False)
        npi = np.where(nanm)[0]
        inn = (npi >= xv[0]) & (npi <= xv[-1])
        if inn.any():
            fixed[npi[inn]] = cs(npi[inn])
        for p in npi[~inn]:
            fixed[p] = fixed[xv[np.argmin(np.abs(xv-p))]]
    for i, f in enumerate(frames):
        if f.get('has_pose') and f.get('landmarks'):
            for lm in f['landmarks']:
                if lm['name']==lm_name and not np.isnan(fixed[i]):
                    lm[axis] = float(fixed[i])
    cnt = int(outlier.sum())
    if cnt > 0:
        print(f"      {lm_name}_{axis}: {cnt}프레임 보정")
    return frames, cnt


def regression_outlier_fix(frames):
    total = 0
    for name in OCCLUSION_PRONE:
        for ax, sig in [('x',SIG_XY),('y',SIG_XY),('z',SIG_Z)]:
            frames, cnt = fix_outliers(frames, name, ax, sig)
            total += cnt
    return frames, total


def compute_bone_lengths(landmarks):
    lmd = {lm['name']: lm for lm in landmarks}
    out = {}
    for p1, p2 in BONE_PAIRS:
        if p1 in lmd and p2 in lmd:
            a, b = lmd[p1], lmd[p2]
            d = math.sqrt((a['x']-b['x'])**2 + (a['y']-b['y'])**2)
            if d > 1e-6:
                out[(p1,p2)] = d
    return out


def enforce_bone_length(frames, ref):
    for f in frames:
        if not f.get('has_pose') or not f.get('landmarks'):
            continue
        lmd = {lm['name']: lm for lm in f['landmarks']}
        for (p1,p2), ref_len in ref.items():
            if p1 not in lmd or p2 not in lmd:
                continue
            a, b = lmd[p1], lmd[p2]
            dx, dy = b['x']-a['x'], b['y']-a['y']
            cur = math.sqrt(dx**2+dy**2)
            if cur < 1e-6:
                continue
            if abs(cur-ref_len)/ref_len > BONE_TOL:
                sc = ref_len/cur
                b['x'] = a['x'] + dx*sc
                b['y'] = a['y'] + dy*sc
    return frames


def get_anchor(frames, address_fi):
    fd = {f['frame']: f for f in frames}
    for fi in ([address_fi] if address_fi is not None else []) + list(fd.keys()):
        f = fd.get(fi)
        if not f or not f.get('has_pose'):
            continue
        hips = {lm['name']: lm for lm in f.get('landmarks',[])
                if lm['name'] in ('left_hip','right_hip')}
        if len(hips) == 2:
            ax = (hips['left_hip']['x'] + hips['right_hip']['x']) / 2
            ay = (hips['left_hip']['y'] + hips['right_hip']['y']) / 2
            return ax, ay, fi
    return None, None, None

print("✅ 완료")

✅ 완료


In [7]:
def process_video_step3(step24_json_path, output_base, view_type):
    out_dir = Path(output_base) / 'step3_unity' / f'z_{Z_SCALE}' / view_type
    out_dir.mkdir(parents=True, exist_ok=True)

    with open(step24_json_path) as f:
        data = json.load(f)

    video_name = data['video']
    frames     = data['frames']
    events     = data.get('events', {})
    addr_info  = events.get('address', {})
    addr_fi    = addr_info.get('frame') if isinstance(addr_info, dict) else addr_info

    print(f"  [{video_name}]  landmark_count={data.get('landmark_count',34)}")

    frames = cubic_spline_interp(frames)
    print(f"    ① Spline 완료")

    frames, total_fixed = regression_outlier_fix(frames)
    print(f"    ② 이상치 보정: {total_fixed}프레임")

    ref_len = {}
    if addr_fi is not None:
        fd = {f['frame']:f for f in frames}.get(addr_fi)
        if fd and fd.get('landmarks'):
            ref_len = compute_bone_lengths(fd['landmarks'])
    if ref_len:
        frames = enforce_bone_length(frames, ref_len)
    print(f"    ③ Bone 정규화: {len(ref_len)}개")

    ax, ay, anchor_fi = get_anchor(frames, addr_fi)
    if ax is None:
        ax, ay = 0.0, 0.0
    print(f"    ④ anchor: frame#{anchor_fi} ({ax:.4f}, {ay:.4f})")

    unity_frames = []
    for fd in frames:
        lms      = fd.get('landmarks', [])
        has_pose = fd.get('has_pose', False)
        processed = ([{**lm,
                       'x':  lm['x'] - ax,
                       'y': -(lm['y'] - ay),
                       'z':  lm['z'] * Z_SCALE}
                      for lm in lms]
                     if has_pose and lms else [])
        unity_frames.append({
            'frame':     fd['frame'],
            'timestamp': fd['timestamp'],
            'has_pose':  has_pose,
            'landmarks': processed,
        })

    out_path = out_dir / f"{video_name}_unity.json"
    with open(out_path, 'w') as f:
        json.dump({
            'video':            video_name,
            'view_type':        view_type,
            'original_size':    data.get('original_size'),
            'fps':              data.get('fps'),
            'total_frames':     data.get('total_frames'),
            'frames_with_pose': sum(1 for f in unity_frames if f['has_pose']),
            'landmark_count':   data.get('landmark_count', 34),
            'landmark_names':   data.get('landmark_names', ALL_34),
            'events':           events,
            'extraction':       data.get('extraction', {}),
            'preprocessing': {
                'cubic_spline':  f'양방향, max_gap={MAX_GAP}',
                'outlier_fix':   f'MAD σ_xy={SIG_XY}, σ_z={SIG_Z}',
                'bone_normalize':f'tolerance={BONE_TOL*100:.0f}%',
                'total_fixed':   total_fixed,
            },
            'conversion': {
                'anchor':       f'address frame#{addr_fi} 골반 중점',
                'y_flip':       'y = -(y - anchor_y)',
                'z_scale':      f'z = z × {Z_SCALE}',
                'anchor_frame': anchor_fi,
                'anchor_xy':    {'x': ax, 'y': ay},
            },
            'frames': unity_frames,
        }, f, indent=2)

    print(f"  ✅ {video_name} → {out_path.name}")
    return str(out_path)

print("✅ 완료")

✅ 완료


In [13]:
from pathlib import Path
step24_dir = Path(DATA_DIR) / 'step24_events'
assert step24_dir.exists(), "❌ step24_events 없음 — 셀 7 먼저 실행"

json_files = []
for view_dir in sorted(step24_dir.iterdir()):
    if view_dir.is_dir():
        for jf in sorted(view_dir.glob('*_events.json')):
            json_files.append({'json': jf, 'view_type': view_dir.name})

print(f"총 {len(json_files)}개\n")

results_3 = []
for i, jf in enumerate(json_files):
    name = jf['json'].stem.replace('_events','')
    print(f"[{i+1}/{len(json_files)}] {name}  [{jf['view_type']}]")
    try:
        out = process_video_step3(jf['json'], BASE_DIR, jf['view_type'])
        results_3.append({'video': name, 'output': out})
    except Exception as e:
        import traceback; traceback.print_exc()
        results_3.append({'video': name, 'error': str(e)})

ok = [r for r in results_3 if 'error' not in r]
print(f"\n{'='*50}")
print(f"✅ Step 3 완료: {len(ok)}/{len(json_files)} 성공")
print(f"출력: {BASE_DIR}/step3_unity/z_{Z_SCALE}/")

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
      right_wrist_y: 61프레임 보정
      right_wrist_z: 61프레임 보정
      left_knee_x: 40프레임 보정
      left_knee_y: 45프레임 보정
      left_knee_z: 43프레임 보정
      left_thumb_x: 51프레임 보정
      left_thumb_y: 63프레임 보정
      left_thumb_z: 53프레임 보정
      right_elbow_x: 60프레임 보정
      right_elbow_y: 78프레임 보정
      right_elbow_z: 57프레임 보정
      left_foot_index_x: 53프레임 보정
      left_foot_index_y: 64프레임 보정
      left_foot_index_z: 35프레임 보정
      right_pinky_x: 56프레임 보정
      right_pinky_y: 67프레임 보정
      right_pinky_z: 58프레임 보정
      right_knee_x: 60프레임 보정
      right_knee_y: 53프레임 보정
      right_knee_z: 44프레임 보정
      left_index_x: 62프레임 보정
      left_index_y: 60프레임 보정
      left_index_z: 47프레임 보정
      right_index_x: 57프레임 보정
      right_index_y: 67프레임 보정
      right_index_z: 60프레임 보정
      left_wrist_x: 33프레임 보정
      left_wrist_y: 58프레임 보정
      left_wrist_z: 46프레임 보정
      right_heel_x: 73프레임 보정
      right_heel_y: 69프레임 보정
      right_heel_z: 48프레임 보정
      left_pi

In [14]:
import glob

unity_files = glob.glob(f'{BASE_DIR}/step3_unity/z_{Z_SCALE}/**/*_unity.json', recursive=True)
print(f"Unity JSON: {len(unity_files)}개\n")

for fp in unity_files[:3]:
    with open(fp) as f:
        d = json.load(f)
    print(f"📄 {d['video']}  [{d['view_type']}]")
    print(f"   landmark_count   : {d['landmark_count']}  ← 34 이어야 함")
    print(f"   total_frames     : {d['total_frames']}")
    print(f"   frames_with_pose : {d['frames_with_pose']}")
    print(f"   events           : { {k:v['frame'] for k,v in d['events'].items()} }")
    fi = next((f for f in d['frames'] if f['has_pose']), None)
    if fi:
        print(f"   첫 포즈 프레임 lm수: {len(fi['landmarks'])}  ← 34 이어야 함")
    print()

Unity JSON: 281개

📄 0_square  [dtl]
   landmark_count   : 34  ← 34 이어야 함
   total_frames     : 138
   frames_with_pose : 134
   events           : {'address': 11, 'toe_up': 33, 'mid_backswing': 56, 'top': 101, 'mid_downswing': 102, 'impact': 103, 'mid_follow_through': 110, 'finish': 117}
   첫 포즈 프레임 lm수: 34  ← 34 이어야 함

📄 1000_square  [dtl]
   landmark_count   : 34  ← 34 이어야 함
   total_frames     : 392
   frames_with_pose : 392
   events           : {'address': 31, 'toe_up': 101, 'mid_backswing': 172, 'top': 314, 'mid_downswing': 319, 'impact': 325, 'mid_follow_through': 332, 'finish': 339}
   첫 포즈 프레임 lm수: 34  ← 34 이어야 함

📄 1001_square  [dtl]
   landmark_count   : 34  ← 34 이어야 함
   total_frames     : 92
   frames_with_pose : 92
   events           : {'address': 7, 'toe_up': 17, 'mid_backswing': 28, 'top': 49, 'mid_downswing': 49, 'impact': 50, 'mid_follow_through': 57, 'finish': 64}
   첫 포즈 프레임 lm수: 34  ← 34 이어야 함

